# Training

Colab notebook for dataset download, VOC-to-YOLO conversion, YOLO11m training, YOLO11s distillation, validation, ONNX export, and ByteTrack FPS.

VOC reference: https://docs.ultralytics.com/datasets/detect/voc


Dataset citation: Ammar, A., Koubaa, A., Ahmed, M., Saad, A. and Benjdira, B., 2021. Vehicle detection from aerial images using deep learning: A comparative study. Electronics, 10(7), p.820.


In [ ]:
%pip install -q -U ultralytics kaggle onnx onnxslim opencv-python

In [ ]:
from pathlib import Path
import json
import os
import random
import shutil
import subprocess
import sys
import time
import xml.etree.ElementTree as ET

PROJECT_ROOT = Path("/content/aerial-car-tracker") if "google.colab" in sys.modules else Path.cwd()
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
os.chdir(PROJECT_ROOT)

SEED = 42
TRAIN_IMGSZ = 1280
EXPORT_IMGSZ = 640
EPOCHS = 200
DATA_DIR = PROJECT_ROOT / "data" / "aerial_cars_yolo"
DATA_YAML = PROJECT_ROOT / "configs" / "aerial_cars.yaml"
RUNS_DIR = PROJECT_ROOT / "runs" / "detect"
REPORTS_DIR = PROJECT_ROOT / "reports"
print(PROJECT_ROOT)

## Download

In [ ]:
from google.colab import userdata

token = userdata.get("KAGGLE_API_TOKEN")
if not token:
    raise RuntimeError("Set KAGGLE_API_TOKEN in Colab Secrets and enable notebook access.")
os.environ["KAGGLE_API_TOKEN"] = token

dataset_dir = PROJECT_ROOT / "dataset"
dataset_dir.mkdir(exist_ok=True)
subprocess.run([
    "kaggle", "datasets", "download", "-d", "riotulab/aerial-images-of-cars",
    "-p", str(dataset_dir), "--unzip"
], check=True)
SRC = dataset_dir / "Aerial-cars"
print(SRC)

## Convert

In [ ]:
def voc_box_to_yolo(width, height, xmin, ymin, xmax, ymax):
    x = ((xmin + xmax) / 2 - 1) / width
    y = ((ymin + ymax) / 2 - 1) / height
    w = (xmax - xmin) / width
    h = (ymax - ymin) / height
    return x, y, w, h


def convert_split(xmls, split):
    image_dir = DATA_DIR / "images" / split
    label_dir = DATA_DIR / "labels" / split
    image_dir.mkdir(parents=True, exist_ok=True)
    label_dir.mkdir(parents=True, exist_ok=True)
    box_count = 0

    for xml_path in xmls:
        root = ET.parse(xml_path).getroot()
        filename = root.findtext("filename")
        size = root.find("size")
        width = int(size.findtext("width"))
        height = int(size.findtext("height"))
        shutil.copy2(xml_path.parent / filename, image_dir / filename)

        rows = []
        for obj in root.findall("object"):
            if obj.findtext("name") != "car" or obj.findtext("difficult", "0") == "1":
                continue
            box = obj.find("bndbox")
            coords = [float(box.findtext(k)) for k in ("xmin", "ymin", "xmax", "ymax")]
            rows.append("0 " + " ".join(f"{v:.8f}" for v in voc_box_to_yolo(width, height, *coords)))

        (label_dir / Path(filename).with_suffix(".txt").name).write_text("\n".join(rows) + "\n")
        box_count += len(rows)

    return {"images": len(xmls), "boxes": box_count}


if DATA_DIR.exists():
    shutil.rmtree(DATA_DIR)

train_xmls = sorted((SRC / "train_PSU").glob("*.xml"))
random.Random(SEED).shuffle(train_xmls)
val_count = round(len(train_xmls) * 0.2)
summary = {
    "train": convert_split(sorted(train_xmls[val_count:]), "train"),
    "val": convert_split(sorted(train_xmls[:val_count]), "val"),
    "test": convert_split(sorted((SRC / "test_PSU").glob("*.xml")), "test"),
}
print(summary)

## Yaml

In [ ]:
DATA_YAML.parent.mkdir(parents=True, exist_ok=True)
DATA_YAML.write_text("\n".join([
    f"path: {DATA_DIR}",
    "train: images/train",
    "val: images/val",
    "test: images/test",
    "",
    "names:",
    "  0: car",
    "",
]))
print(DATA_YAML.read_text())

## Teacher

In [ ]:
from ultralytics import YOLO

teacher_result = YOLO("yolo11m.pt").train(
    data=str(DATA_YAML), imgsz=TRAIN_IMGSZ, epochs=EPOCHS, patience=40, batch=-1,
    amp=True, seed=SEED, project=str(RUNS_DIR), name="teacher_yolo11m", exist_ok=True,
)
TEACHER = Path(teacher_result.save_dir) / "weights" / "best.pt"
print(TEACHER)

## Student

In [ ]:
student_result = YOLO("yolo11s.pt").train(
    data=str(DATA_YAML), imgsz=TRAIN_IMGSZ, epochs=EPOCHS, patience=40, batch=-1,
    amp=True, seed=SEED, distill_model=str(TEACHER), dis=6.0,
    project=str(RUNS_DIR), name="student_yolo11s_distill", exist_ok=True,
)
STUDENT = Path(student_result.save_dir) / "weights" / "best.pt"
print(STUDENT)

## Evaluate

In [ ]:
## Evaluate

import json

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from ultralytics import YOLO


# Evaluate the final student model on the reserved test set at the
# same input resolution used for ONNX export and deployment.
model = YOLO(str(STUDENT))

metrics = model.val(
    data=str(DATA_YAML),
    imgsz=EXPORT_IMGSZ,
    split="test",
    project=str(RUNS_DIR),
    name="student_test_640",
    exist_ok=True,
)

report = {
    "split": "test",
    "imgsz": EXPORT_IMGSZ,
    "mAP50": float(metrics.box.map50),
    "mAP50-95": float(metrics.box.map),
    "precision": float(metrics.box.mp),
    "recall": float(metrics.box.mr),
}

REPORTS_DIR.mkdir(parents=True, exist_ok=True)

metrics_json_path = REPORTS_DIR / "detection_metrics.json"
metrics_json_path.write_text(
    json.dumps(report, indent=2),
    encoding="utf-8",
)

metrics_df = pd.DataFrame([report])
display(metrics_df.round(4))

plot_metrics = {
    "mAP50": report["mAP50"],
    "mAP50-95": report["mAP50-95"],
    "Precision": report["precision"],
    "Recall": report["recall"],
}

plt.figure(figsize=(6, 4))
plt.bar(plot_metrics.keys(), plot_metrics.values())
plt.ylim(0, 1)
plt.ylabel("Score")
plt.title(f"Test detection metrics at {EXPORT_IMGSZ}×{EXPORT_IMGSZ}")
plt.tight_layout()

metrics_plot_path = REPORTS_DIR / "detection_metrics.png"
plt.savefig(metrics_plot_path, dpi=150, bbox_inches="tight")
plt.show()

print(f"Metrics saved to: {metrics_json_path}")
print(f"Plot saved to: {metrics_plot_path}")

## Export

In [ ]:
export_script = PROJECT_ROOT / "export_yolo11.py"
if not export_script.exists():
    subprocess.run([
        "curl", "-L",
        "https://raw.githubusercontent.com/marcoslucianops/DeepStream-Yolo/master/utils/export_yolo11.py",
        "-o", str(export_script),
    ], check=True)

subprocess.run([
    sys.executable, str(export_script),
    "-w", str(STUDENT),
    "-s", str(EXPORT_IMGSZ),
    "--opset", "12",
    "--simplify",
], check=True)

bundle = PROJECT_ROOT / "exports" / "training_output"
bundle.mkdir(parents=True, exist_ok=True)
shutil.copy2(STUDENT.with_suffix(".onnx"), bundle / "yolo11s_car.onnx")
shutil.copy2(PROJECT_ROOT / "labels.txt", bundle / "labels.txt")
shutil.copy2(REPORTS_DIR / "detection_metrics.json", bundle / "detection_metrics.json")
shutil.copy2(REPORTS_DIR / "detection_metrics.png", bundle / "detection_metrics.png")
print(bundle)

## Bytetrack FPS

In [ ]:
VIDEO_SOURCE = ""  # e.g. "/content/video.mp4"

if VIDEO_SOURCE:
    frames = 0
    start = time.perf_counter()
    for _ in model.track(source=VIDEO_SOURCE, tracker="bytetrack.yaml", stream=True, imgsz=EXPORT_IMGSZ, verbose=False):
        frames += 1
    fps = frames / (time.perf_counter() - start)
    print({"frames": frames, "fps": fps})